# Introducción a QA sobre noticias dominicanas

Recursos oficiales:

- Dataset: `Lisibonny/pdqa`
- Baseline: `Lisibonny/modelo_qa_beto_squad_es_pdqa`
- Space: `Lisibonny/Repartidor_Dominicano`

El dataset ya contiene las divisiones oficiales: `train`, `validation` y `test`.


In [1]:
!pip install -q "transformers>=4.45,<5.0" \
                "datasets>=3.0,<5.0" \
                "accelerate>=1.0,<2.0" \
                "huggingface_hub>=0.28,<2.0" \
                "torch>=2.2" \
                "pandas>=2.0,<3.0" \
                "numpy>=1.26,<3.0" \
                "pyarrow>=15,<22" \
                "scikit-learn>=1.4,<2.0"

In [2]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

In [3]:
import random, string, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
import json

SEED = 42
DATASET_ID = "Lisibonny/pdqa"
BASELINE_MODEL_ID = "Lisibonny/modelo_qa_beto_squad_es_pdqa"
FT_BASE_MODEL_ID = "nlp-en-es/roberta-base-bne-finetuned-sqac"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

dataset = load_dataset(DATASET_ID)
print(dataset)
assert set(["train","validation","test"]).issubset(dataset.keys())
print({split: len(dataset[split]) for split in dataset})


DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'context', 'answers', 'title'],
        num_rows: 45
    })
    validation: Dataset({
        features: ['id', 'question', 'context', 'answers', 'title'],
        num_rows: 15
    })
    test: Dataset({
        features: ['id', 'question', 'context', 'answers', 'title'],
        num_rows: 20
    })
})
{'train': 45, 'validation': 15, 'test': 20}


In [4]:
print(dataset["train"].column_names)
display(pd.DataFrame([dataset["train"][0]]))
display(pd.DataFrame([dataset["validation"][0]]))
display(pd.DataFrame([dataset["test"][0]]))


['id', 'question', 'context', 'answers', 'title']


,id,question,context,answers,title
0,5kpfplqtw2p99k1,¿Quiénes se van de “De Extremo a Extremo”?,Durante la pasada entrega de Premios Soberano ...,"{'answer_start': [171], 'text': ['Caroline Aqu...",prueba


,id,question,context,answers,title
0,c61s2xwyfi0cugi,¿Cuál es el objetivo de la campaña de Asindown?,Asindown ha lanzado una campaña de sensibiliza...,"{'answer_start': [161], 'text': ['concienciar ...",prueba


,id,question,context,answers,title
0,jygfnzhtybqz9u2,¿Quién escribió el poema “Una mujer está sola”?,"“Una mujer está sola”, escribió Aída Portalatí...","{'answer_start': [32], 'text': ['Aída Portalat...",prueba


## Verificación de columnas

El notebook espera columnas equivalentes a `id`, `question`, `context` y `answers`.
Si los nombres son distintos, ajuste solo las constantes siguientes.


In [5]:
ID_COL = "id"
QUESTION_COL = "question"
CONTEXT_COL = "context"
ANSWERS_COL = "answers"

required_train = {ID_COL, QUESTION_COL, CONTEXT_COL, ANSWERS_COL}
required_test = {ID_COL, QUESTION_COL, CONTEXT_COL}
assert required_train.issubset(dataset["train"].column_names)
assert required_train.issubset(dataset["validation"].column_names)
assert required_test.issubset(dataset["test"].column_names)


## Análisis exploratorio mínimo

In [6]:
train_df = dataset["train"].to_pandas()
validation_df = dataset["validation"].to_pandas()

for frame in [train_df, validation_df]:
    frame["question_words"] = frame[QUESTION_COL].astype(str).str.split().str.len()
    frame["context_words"] = frame[CONTEXT_COL].astype(str).str.split().str.len()

display(train_df[["question_words","context_words"]].describe())
display(validation_df[["question_words","context_words"]].describe())


,question_words,context_words
count,45.000000,45.000000
mean,6.244444,60.644444
std,2.612518,56.302412
min,3.000000,27.000000
25%,4.000000,34.000000
50%,6.000000,43.000000
75%,7.000000,55.000000
max,15.000000,285.000000


,question_words,context_words
count,15.000000,15.000000
mean,5.933333,98.133333
std,1.667619,112.981583
min,3.000000,29.000000
25%,5.000000,34.000000
50%,6.000000,38.000000
75%,6.000000,87.000000
max,9.000000,347.000000


## Inferencia con el baseline

In [7]:
device = 0 if torch.cuda.is_available() else -1
qa_baseline = pipeline(
    "question-answering",
    model=BASELINE_MODEL_ID,
    tokenizer=BASELINE_MODEL_ID,
    device=device,
)

example = dataset["validation"][0]
result = qa_baseline(
    question=example[QUESTION_COL],
    context=example[CONTEXT_COL],
)
print("Pregunta:", example[QUESTION_COL])
print("Referencia:", example[ANSWERS_COL])
print("Predicción:", result)


Device set to use cuda:0


Pregunta: ¿Cuál es el objetivo de la campaña de Asindown?
Referencia: {'answer_start': [161], 'text': ['concienciar sobre el acoso escolar y social que sufren las personas con síndrome de Down']}
Predicción: {'score': 0.09938068687915802, 'start': 161, 'end': 204, 'answer': 'concienciar sobre el acoso escolar y social'}


## Exact Match y F1

In [8]:
SPANISH_ARTICLES = {"el","la","los","las","un","una","unos","unas"}

def normalize_answer(text):
    text = "" if text is None else str(text).lower().strip()
    text = "".join(ch for ch in unicodedata.normalize("NFD", text)
                   if unicodedata.category(ch) != "Mn")
    punctuation = string.punctuation + "¡¿“”‘’«»…"
    text = "".join(" " if ch in punctuation else ch for ch in text)
    return " ".join(tok for tok in text.split() if tok not in SPANISH_ARTICLES)

def exact_match(pred, truth):
    return float(normalize_answer(pred) == normalize_answer(truth))

def token_f1(pred, truth):
    p = normalize_answer(pred).split()
    t = normalize_answer(truth).split()
    if not p and not t: return 1.0
    if not p or not t: return 0.0
    same = sum((Counter(p) & Counter(t)).values())
    if same == 0: return 0.0
    precision = same / len(p)
    recall = same / len(t)
    return 2 * precision * recall / (precision + recall)

def answer_texts(value):
    if isinstance(value, dict):
        value = value.get("text", value.get("answer", value))
    if isinstance(value, list):
        return [str(x) for x in value]
    return [str(value)]


In [9]:
rows = []
for ex in dataset["validation"]:
    pred = qa_baseline(question=ex[QUESTION_COL], context=ex[CONTEXT_COL])
    golds = answer_texts(ex[ANSWERS_COL])
    em = max(exact_match(pred["answer"], g) for g in golds)
    f1 = max(token_f1(pred["answer"], g) for g in golds)
    rows.append({
        "id": ex[ID_COL],
        "question": ex[QUESTION_COL],
        "reference": " | ".join(golds),
        "prediction": pred["answer"],
        "confidence": pred["score"],
        "exact_match": em,
        "f1": f1,
    })

baseline_results = pd.DataFrame(rows)
display(baseline_results)
print("Exact Match:", 100 * baseline_results["exact_match"].mean())
print("F1:", 100 * baseline_results["f1"].mean())


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,id,question,reference,prediction,confidence,exact_match,f1
0,c61s2xwyfi0cugi,¿Cuál es el objetivo de la campaña de Asindown?,concienciar sobre el acoso escolar y social qu...,concienciar sobre el acoso escolar y social,0.099381,0.0,0.631579
1,7857zba14eers9x,¿Cómo se llama el canal 4?,Radio Televisión Dominicana,Radio Televisión Dominicana,0.634700,1.0,1.000000
2,hkhjremdmrmix29,¿Cuál es nuestro símbolo patrio?,La Bandera Nacional,La Bandera Nacional,0.382562,1.0,1.000000
3,ayfu15q9bfn5nhb,¿Quién fundó Menudo?,Edgardo García,Edgardo García,0.545798,1.0,1.000000
4,rpcbxp9rc8g0iuh,¿Cuando es el Miércoles de Ceniza?,El próximo miércoles 22 de febrero,El próximo miércoles 22 de febrero,0.336815,1.0,1.000000
5,d2xt90fah10kox5,¿A cuántos bateadores ponchó Jacob deGrom?,a 11 bateadores,11 bateadores,0.367446,0.0,0.800000
6,sz9yeioprmn61xv,¿Quién es el creador de Dilbert?,Scott Adams,Scott Adams,0.959120,1.0,1.000000
7,ak5ubvypoj83wfp,¿Cuántos minutos jugó Immanuel Quickley?,55 minutos,55 minutos,0.447666,1.0,1.000000
8,3u77e1org81bc7f,¿Qué jugador estaba lesionado?,Jalen Brunson,Jalen Brunson,0.796943,1.0,1.000000
9,pxqbvfcey7z9p3q,¿Cómo será la inflación en 2023?,seguirá siendo alta,en torno al 7%,0.187205,0.0,0.000000


Exact Match: 60.0
F1: 75.84015594541911


## Fine-tuning para mejorar Exact Match


In [10]:
# Importar librerías
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer, default_data_collator
from transformers import TrainerCallback

### Alineación de `answer_start`

Antes de entrenar, verificamos que el offset de cada respuesta coincida con
el texto real del `context`. Si no coincide, el ejemplo se descartaría
silenciosamente en el preprocesamiento (quedaría con `start_position=0`).

In [11]:
# Función para extraer el texto de la respuesta y su posición inicial del dataset
def get_answer_start_text(ans):
    if isinstance(ans, dict):
        text = ans["text"][0] if isinstance(ans.get("text"), list) else ans.get("text", ans.get("answer"))
        start = ans["answer_start"][0] if isinstance(ans.get("answer_start"), list) else ans.get("answer_start")
        return str(text), start
    raise ValueError(f"Formato de answers no soportado: {ans}")

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

Con esta función se obtiene el texto de la respuesta y la posición donde empieza dentro del contexto, sin importar cómo estén organizados esos datos en el dataset. Si el texto o `answer_start` vienen en una lista, toma el primer elemento; de lo contrario, usa el valor directamente.

Al final, devuelve el texto de la respuesta junto con su posición inicial. Si el formato de `answers` no es el esperado, genera un error para evitar problemas durante el procesamiento de los datos.
</div>

In [12]:
# Función para reemplazar espacios no separables por espacios normales
def normalize_spaces(text):
    return text.replace("\xa0", " ")

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

La función `normalize_spaces()` se utiliza para reemplazar los espacios no separables (`\xa0`) por espacios normales. Esto permite que las comparaciones entre el texto de la respuesta y el fragmento del contexto sean correctas, evitando diferencias que solo se deben al formato del texto y no al contenido.
</div>

In [13]:
# Función para verificar que las respuestas estén alineadas con el contexto
def check_alignment(hf_split, split_name):
    problems = []
    for ex in hf_split:
        context = ex[CONTEXT_COL]
        try:
            answer_text, char_start = get_answer_start_text(ex[ANSWERS_COL])
        except Exception as exc:
            problems.append({"id": ex[ID_COL], "issue": f"formato answers inesperado: {exc}"})
            continue
        if char_start is None:
            problems.append({"id": ex[ID_COL], "issue": "answer_start es None", "answer_text": answer_text})
            continue
        char_end = char_start + len(answer_text)
        slice_from_context = context[char_start:char_end]
        if normalize_spaces(slice_from_context) != normalize_spaces(answer_text):
            found_at = context.find(answer_text)
            problems.append({
                "id": ex[ID_COL],
                "issue": "desalineado",
                "answer_text": answer_text,
                "answer_start_dado": char_start,
                "texto_en_ese_offset": slice_from_context,
                "offset_correcto_si_se_encontro": found_at,
            })
    print(f"{split_name}: {len(problems)} / {len(hf_split)} ejemplos con problemas de alineación")
    return pd.DataFrame(problems)

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

Con esta función se comprueba que cada respuesta del dataset realmente se encuentre en la posición indicada por `answer_start` dentro del contexto. Antes de comparar ambos textos, se normalizan los espacios para evitar diferencias causadas por el formato, como los espacios no separables (`\xa0`).

Al final, muestra cuántos ejemplos tienen problemas de alineación y devuelve un `DataFrame` con esos casos. Esto ayuda a comprobar que los datos estén correctos antes de entrenar el modelo.
</div>

## Ejecutar la verificación en train y validation

In [14]:
train_alignment_issues = check_alignment(dataset["train"], "train")
val_alignment_issues = check_alignment(dataset["validation"], "validation")

train: 0 / 45 ejemplos con problemas de alineación
validation: 0 / 15 ejemplos con problemas de alineación


<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

Se verifica que las respuestas de los conjuntos de entrenamiento y validación estén correctamente alineadas con su contexto. Como resultado, ningún ejemplo presentó problemas de alineación, ya que tanto el conjunto de entrenamiento como el de validación obtuvieron **0 casos con errores**.

Esto indica que los datos son consistentes y que la posición indicada por `answer_start` coincide con el texto de la respuesta en todos los ejemplos.
</div>

In [15]:
if len(train_alignment_issues):
    display(train_alignment_issues)
if len(val_alignment_issues):
    display(val_alignment_issues)

if len(train_alignment_issues) == 0 and len(val_alignment_issues) == 0:
    print("Todo alineado correctamente")

Todo alineado correctamente


<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

Se comprueba si existen ejemplos con problemas de alineación. Como no se detectó ningún caso en los conjuntos de entrenamiento y validación, se muestra el mensaje **"Todo alineado correctamente"**, confirmando que los datos pueden utilizarse para entrenar el modelo.
</div>

## Preparación de los datos para el entrenamiento

In [16]:
MAX_LENGTH = 384
DOC_STRIDE = 128

# Cargar el tokenizador del modelo base
tokenizer = AutoTokenizer.from_pretrained(FT_BASE_MODEL_ID)

# Función para preparar los datos de entrenamiento del modelo
def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples[QUESTION_COL]]
    tokenized = tokenizer(
        questions,
        examples[CONTEXT_COL],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer_text, char_start = get_answer_start_text(examples[ANSWERS_COL][sample_idx])
        char_end = char_start + len(answer_text)

        sequence_ids = tokenized.sequence_ids(i)
        ctx_start = sequence_ids.index(1)
        ctx_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        if offsets[ctx_start][0] > char_start or offsets[ctx_end][1] < char_end:
            start_positions.append(0)
            end_positions.append(0)
            continue

        idx = ctx_start
        while idx <= ctx_end and offsets[idx][0] <= char_start:
            idx += 1
        start_positions.append(idx - 1)

        idx = ctx_end
        while idx >= ctx_start and offsets[idx][1] >= char_end:
            idx -= 1
        end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

train_features = dataset["train"].map(
    prepare_train_features, batched=True, remove_columns=dataset["train"].column_names
)

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

En esta parte se preparan los datos para poder entrenar el modelo. Primero se carga el tokenizador y se define la cantidad máxima de tokens que tendrá cada ejemplo, además de un solapamiento para los contextos que sean muy largos.

Luego, la función `prepare_train_features()` convierte las preguntas y los contextos en tokens y calcula la posición donde empieza y termina cada respuesta. Si una respuesta no cabe completa dentro del fragmento del contexto, se le asignan las posiciones `(0, 0)` para que no sea tomada en cuenta.

Finalmente, se aplica esta función a todo el conjunto de entrenamiento utilizando `map()`, obteniendo los datos que se usarán para entrenar el modelo.
</div>

In [17]:
print(train_features)

Dataset({
    features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 47
})


<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

Se muestra el conjunto de datos después del preprocesamiento. Se puede observar que contiene los atributos `input_ids`, `attention_mask`, `start_positions` y `end_positions`, que son los datos que necesita el modelo para aprender durante el entrenamiento.

Además, el conjunto cuenta con **47 ejemplos**. Este número es mayor que los **45 ejemplos originales**, ya que algunos contextos largos fueron divididos en varios fragmentos durante el proceso de tokenización utilizando el parámetro `stride`.
</div>

In [18]:
# Conjunto de combinaciones de hiperparámetros para probar durante el entrenamiento
HYPERPARAM_GRID = [
    {"learning_rate": 3e-5, "num_train_epochs": 12, "batch_size": 4, "weight_decay": 0.01},
    {"learning_rate": 2e-5, "num_train_epochs": 12, "batch_size": 4, "weight_decay": 0.05},
    {"learning_rate": 1e-5, "num_train_epochs": 15, "batch_size": 8, "weight_decay": 0.05},
]

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

En esta celda se define un conjunto de combinaciones de hiperparámetros que se utilizarán para entrenar el modelo. Cada combinación incluye valores para la tasa de aprendizaje (`learning_rate`), el número de épocas (`num_train_epochs`), el tamaño del lote (`batch_size`) y la regularización (`weight_decay`).

El objetivo es probar diferentes configuraciones y comparar sus resultados para identificar cuál ofrece el mejor desempeño durante el entrenamiento.
</div>

In [19]:
# Función para evaluar el desempeño del modelo en un conjunto de datos
def evaluate_split_ft(qa_pipe, hf_split):
    rows = []
    for ex in hf_split:
        pred = qa_pipe(question=ex[QUESTION_COL], context=ex[CONTEXT_COL])
        golds = answer_texts(ex[ANSWERS_COL])
        em = max(exact_match(pred["answer"], g) for g in golds)
        f1 = max(token_f1(pred["answer"], g) for g in golds)
        rows.append({"id": ex[ID_COL], "prediction": pred["answer"], "exact_match": em, "f1": f1})
    df = pd.DataFrame(rows)
    return 100 * df["exact_match"].mean(), 100 * df["f1"].mean(), df

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

En esta función se evalúa el desempeño del modelo utilizando un conjunto de datos. Para cada ejemplo, el modelo genera una respuesta a partir de la pregunta y el contexto, y luego esta respuesta se compara con la respuesta correcta del dataset.

Después, se calculan las métricas **Exact Match (EM)** y **F1** para cada ejemplo. Finalmente, se obtiene el promedio de ambas métricas y se devuelve junto con un `DataFrame` que contiene las predicciones y los resultados obtenidos en cada caso.
</div>

In [20]:
# Clase para evaluar el modelo al final de cada época y guardar el que obtenga el mejor resultado
class EMEarlyStoppingCallback(TrainerCallback):

    def __init__(self, trainer_ref, tokenizer, val_split):
        self.trainer_ref = trainer_ref
        self.tokenizer = tokenizer
        self.val_split = val_split
        self.best_em = -1
        self.best_f1 = -1
        self.best_epoch = None
        self.best_state_dict = None
        self.history = []

    def on_epoch_end(self, args, state, control, **kwargs):
        model = self.trainer_ref.model
        model.eval()
        qa_pipe = pipeline("question-answering", model=model, tokenizer=self.tokenizer, device=device)
        em, f1, _ = evaluate_split_ft(qa_pipe, self.val_split)
        epoch = round(state.epoch)
        self.history.append({"epoch": epoch, "exact_match": em, "f1": f1})
        print(f"   [epoch {epoch}] EM={em:.2f}  F1={f1:.2f}")
        if (em > self.best_em) or (em == self.best_em and f1 > self.best_f1):
            self.best_em, self.best_f1, self.best_epoch = em, f1, epoch
            self.best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        model.train()

<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

En esta parte se crea una clase que permite evaluar el modelo al finalizar cada época de entrenamiento. Para ello, se calculan las métricas **Exact Match (EM)** y **F1** utilizando el conjunto de validación.

Si el modelo obtiene un mejor resultado que en las épocas anteriores, se guardan sus pesos, junto con la época y las métricas obtenidas. De esta manera, al finalizar el entrenamiento es posible conservar la versión del modelo que presentó el mejor desempeño durante la validación.
</div>

## Entrenamiento

In [21]:
#Entrenar el modelo con diferentes configuraciones de hiperparámetros y seleccionar la que obtenga el mejor resultado en validación
results_log = []
best = {"em": -1, "config": None, "output_dir": None}

for run_idx, cfg in enumerate(HYPERPARAM_GRID):
    print(f"\n=== Run {run_idx}: {cfg} ===")
    torch.manual_seed(SEED)
    model = AutoModelForQuestionAnswering.from_pretrained(FT_BASE_MODEL_ID)
    out_dir = f"./run_{run_idx}"

    args = TrainingArguments(
        output_dir=out_dir,
        learning_rate=cfg["learning_rate"],
        num_train_epochs=cfg["num_train_epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        weight_decay=cfg["weight_decay"],
        seed=SEED,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_features,
        data_collator=default_data_collator,
        tokenizer=tokenizer,
    )
    callback = EMEarlyStoppingCallback(trainer, tokenizer, dataset["validation"])
    trainer.add_callback(callback)
    trainer.train()

    model.load_state_dict(callback.best_state_dict)
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    print(f"Run {run_idx} -> mejor época: {callback.best_epoch}  EM: {callback.best_em:.2f}  F1: {callback.best_f1:.2f}")
    results_log.append({**cfg, "best_epoch": callback.best_epoch, "exact_match": callback.best_em,
                         "f1": callback.best_f1, "output_dir": out_dir})

    qa_pipe = pipeline("question-answering", model=out_dir, tokenizer=out_dir, device=device)
    em, f1, detail_df = evaluate_split_ft(qa_pipe, dataset["validation"])

    if em > best["em"] or (em == best["em"] and f1 > best.get("f1", -1)):
        best = {"em": em, "f1": f1, "config": cfg, "output_dir": out_dir, "detail_df": detail_df,
                "best_epoch": callback.best_epoch}

results_df = pd.DataFrame(results_log)
display(results_df)
print("\nMejor configuración por EM en validation:", best["config"],
      "| mejor época:", best["best_epoch"], "| EM:", best["em"], "| F1:", best["f1"])


=== Run 0: {'learning_rate': 3e-05, 'num_train_epochs': 12, 'batch_size': 4, 'weight_decay': 0.01} ===


/tmp/ipykernel_25756/1978737380.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
5,1.653600
10,1.271100
15,0.769600
20,0.743900
25,0.275500
30,0.079100
35,0.868600
40,0.678600
45,0.080700
50,0.195100


Device set to use cuda:0


   [epoch 1] EM=66.67  F1=83.84


...

Device set to use cuda:0


   [epoch 15] EM=60.00  F1=81.64
Run 2 -> mejor época: 1  EM: 60.00  F1: 81.64


Device set to use cuda:0


,learning_rate,num_train_epochs,batch_size,weight_decay,best_epoch,exact_match,f1,output_dir
0,0.00003,12,4,0.01,1,66.666667,83.841270,./run_0
1,0.00002,12,4,0.05,1,73.333333,84.793651,./run_1
2,0.00001,15,8,0.05,1,60.000000,81.642136,./run_2



Mejor configuración por EM en validation: {'learning_rate': 2e-05, 'num_train_epochs': 12, 'batch_size': 4, 'weight_decay': 0.05} | mejor época: 1 | EM: 73.33333333333333 | F1: 84.79365079365078


<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

En esta etapa se probaron las tres configuraciones de hiperparámetros para identificar cuál obtenía los mejores resultados en el conjunto de validación. Después de entrenar cada modelo, se evaluó utilizando las métricas **Exact Match (EM)** y **F1**.

Al comparar los resultados, la segunda configuración fue la que obtuvo el mejor desempeño, alcanzando un **Exact Match de 73.33%** y un **F1 de 84.79%**. Aunque la primera configuración obtuvo un F1 muy parecido (**83.84%**), su Exact Match fue menor (**66.67%**). Como el criterio principal para seleccionar el modelo fue el valor de **Exact Match**, se eligió la segunda configuración, ya que logró responder correctamente un mayor número de preguntas.

Por otro lado, la tercera configuración fue la que obtuvo el rendimiento más bajo, con un **Exact Match de 60%** y un **F1 de 81.64%**, por lo que no fue considerada como la mejor opción.

También se puede observar que, en las tres configuraciones, el mejor resultado se obtuvo durante la **primera época**. Después de ese punto las métricas dejaron de mejorar e incluso disminuyeron en algunos casos, lo que indica que seguir entrenando no aportaba beneficios y podía hacer que el modelo comenzara a sobreajustarse a los datos de entrenamiento. Por esta razón, se conservaron los pesos correspondientes a la primera época para cada entrenamiento.
</div>

In [22]:
display(best["detail_df"])

,id,prediction,exact_match,f1
0,c61s2xwyfi0cugi,concienciar sobre el acoso escolar y social qu...,1.0,1.000000
1,7857zba14eers9x,Radio Televisión Dominicana,1.0,1.000000
2,hkhjremdmrmix29,La Bandera Nacional,1.0,1.000000
3,ayfu15q9bfn5nhb,Edgardo García,1.0,1.000000
4,rpcbxp9rc8g0iuh,El próximo miércoles 22 de febrero,1.0,1.000000
5,d2xt90fah10kox5,a 11 bateadores en seis entradas,0.0,0.666667
6,sz9yeioprmn61xv,Scott Adams,1.0,1.000000
7,ak5ubvypoj83wfp,55 minutos,1.0,1.000000
8,3u77e1org81bc7f,Jalen Brunson,1.0,1.000000
9,pxqbvfcey7z9p3q,en torno al 7% a nivel mundial,0.0,0.000000


<div style="background:#FFFFE0;padding:20px;color:#000000;margin-top:10px;">

En esta tabla se muestran las predicciones realizadas por el mejor modelo sobre el conjunto de validación, junto con las métricas **Exact Match (EM)** y **F1** para cada ejemplo.

Se puede observar que el modelo respondió correctamente la mayoría de las preguntas, ya que **11 de los 15 ejemplos obtuvieron un Exact Match igual a 1**, lo que significa que la respuesta generada coincide exactamente con la respuesta esperada. En los demás casos, aunque el Exact Match fue 0, algunos ejemplos obtuvieron un valor de F1 relativamente alto, como **0.67** y **0.95**, indicando que la respuesta fue parcialmente correcta y compartía gran parte de las palabras con la respuesta real.

Sin embargo, también hubo casos donde el desempeño fue bajo, como el ejemplo con un F1 de 0, lo que indica que la respuesta generada no coincidió con la respuesta esperada. En general, estos resultados muestran que el modelo fue capaz de responder correctamente la mayoría de las preguntas, aunque todavía presenta algunas dificultades en ciertos ejemplos más complejos.
</div>

## Validación oficial con 05_evaluate_qa.py


In [23]:
# Predicciones del mejor modelo sobre validation
best["detail_df"][["id", "prediction"]].to_csv(
    "val_predictions.csv", index=False, encoding="utf-8-sig"
)

# Referencias oficiales de validation, en el formato que espera 05_evaluate_qa.py
val_refs = [
    {"id": ex[ID_COL], "answers": answer_texts(ex[ANSWERS_COL])}
    for ex in dataset["validation"]
]
with open("val_references.json", "w", encoding="utf-8") as f:
    json.dump(val_refs, f, ensure_ascii=False)

print("val_predictions.csv y val_references.json generados.")

val_predictions.csv y val_references.json generados.


In [24]:
!python 05_evaluate_qa.py --references val_references.json --predictions val_predictions.csv --output val_metrics.json

{
  "exact_match": 73.3333,
  "f1": 84.7937,
  "num_references": 15,
  "num_predictions": 15,
  "missing_ids_count": 0,
  "extra_ids_count": 0,
  "missing_ids": [],
  "extra_ids": []
}


## Subir el modelo a Hugging Face


In [25]:
REPO_ID = "scarletabreu/pdqa-roberta-bne-em-tuned"

# Cargamos los pesos guardados del mejor checkpoint
final_model = AutoModelForQuestionAnswering.from_pretrained(best["output_dir"])
final_tokenizer = AutoTokenizer.from_pretrained(best["output_dir"])

# Subir el modelo y el tokenizador al Hub
final_model.push_to_hub(REPO_ID)
final_tokenizer.push_to_hub(REPO_ID)

print(f"Modelo disponible en: https://huggingface.co/{REPO_ID}")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...upovpkv/model.safetensors:  10%|9         | 48.0MB /  496MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Modelo disponible en: https://huggingface.co/scarletabreu/pdqa-roberta-bne-em-tuned


## Generar submission.csv con el modelo elegido

Cambie `FINAL_MODEL_ID` por el repositorio del modelo final del equipo.


In [26]:
FINAL_MODEL_ID = "scarletabreu/pdqa-roberta-bne-em-tuned"
qa_final = pipeline(
    "question-answering",
    model=FINAL_MODEL_ID,
    tokenizer=FINAL_MODEL_ID,
    device=device,
)

submission_rows = []
for ex in dataset["test"]:
    pred = qa_final(question=ex[QUESTION_COL], context=ex[CONTEXT_COL])
    submission_rows.append({
        "id": ex[ID_COL],
        "prediction": pred["answer"]
    })

submission = pd.DataFrame(submission_rows)
submission.to_csv("submission.csv", index=False, encoding="utf-8-sig")
display(submission)

Device set to use cuda:0


,id,prediction
0,jygfnzhtybqz9u2,Aída Portalatín
1,9zptnl85o2ocses,abraza la violencia a la mujer
2,gm0qmv6gv0qh1hu,Partido de la Liberación Dominicana
3,m9fzivkm3yoslwg,Al menos 4 diputados del Partido de la Liberac...
4,elvla3nmkr4u3rk,frente al Palacio de Justicia de Ciudad Nueva
5,4lp1ayfq8cavvo8,Anuel AA
6,jct9x3le3n2oc7m,por la divulgación de contenido para adultos e...
7,7wfhp30e28oxep5,como orientadora de los estudiantes del liceo ...
8,xnk6ge45wkyul3m,artista urbano puertorriqueño
9,j0wjntn0ot11fpr,en el 112 Dyckman Street en Manhattan


## Lista de comprobación

- [ ] Evalué el baseline en validation.
- [ ] Comparé al menos dos variantes.
- [ ] Registré hiperparámetros y semillas.
- [ ] Realicé una ablación o comparación controlada.
- [ ] Analicé errores.
- [ ] Generé y validé submission.csv.
- [ ] Publiqué el modelo y completé la model card.
